# Partition Cell Types for Clustering and Harmonization
To ensure that our cell type labels are as accurate as possible, we'll subset our dataset based on our AIFI_L3 labels for harmonization and clustering

## Load libraries

In [1]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=RuntimeWarning)

import concurrent.futures
from concurrent.futures import ThreadPoolExecutor
from datetime import date
import hisepy
import os
import pandas as pd
import re
import scanpy as sc
import scanpy.external as sce

In [2]:
out_dir = 'output'
if not os.path.isdir(out_dir):
    os.makedirs(out_dir)

In [27]:
### slice indices for L3 cell types
start = 20
end = 50

## Helper functions

These make it a bit simpler to cache and read in files from HISE

In [4]:
def cache_uuid_path(uuid):
    cache_path = '/home/jupyter/cache/{u}'.format(u = uuid)
    if not os.path.isdir(cache_path):
        hise_res = hisepy.reader.cache_files([uuid])
    filename = os.listdir(cache_path)[0]
    cache_file = '{p}/{f}'.format(p = cache_path, f = filename)
    return cache_file

In [5]:
def read_parquet_uuid(uuid):
    cache_file = cache_uuid_path(uuid)
    res = pd.read_parquet(cache_file)
    return res

In [6]:
def sort_adata_uuid(uuid, sort_cols = ['AIFI_L2', 'sample.sampleKitGuid']):
    cache_file = cache_uuid_path(uuid)
    adata = sc.read_h5ad(cache_file)
    obs = adata.obs
    obs = obs.sort_values(sort_cols)
    adata = adata[obs.index]
    adata.write_h5ad(cache_file)

This function will enable us to connect to our .h5ad files without loading the entire thing into memory. We'll then load only the cells that we want for each cell class to assemble them for writing. This should save us some overhead as we do our subsetting.

In [7]:
def read_adata_backed_uuid(uuid):
    cache_file = cache_uuid_path(uuid)
    res = sc.read_h5ad(cache_file, backed = 'r')
    return res

This function will remove Ig-related genes, which is recommended by Marla Glass for analysis of B cell subtypes

In [8]:
def remove_ig_genes(adata):
    igl_genes = [gene for gene in adata.var_names if gene.startswith("IGL")]
    igk_genes = [gene for gene in adata.var_names if gene.startswith("IGK")]
    ighc_genes = [gene for gene in adata.var_names if gene.startswith("IGH")]
    exl_genes = igl_genes + igk_genes + ighc_genes

    filtered_genes = [gene for gene in adata.var_names if gene not in exl_genes]
    adata = adata[:, filtered_genes]

    return adata

This function will apply a standard normalization, nearest neighbors, clustering, and UMAP process to our cell subsets:

In [9]:
def process_adata(adata, resolution = 2):
    
    # Keep a copy of the raw data
    adata = adata.raw.to_adata()
    adata.raw = adata

    print('Normalizing', end = "; ")
    # Normalize and log transform
    sc.pp.normalize_total(adata)
    sc.pp.log1p(adata)

    print('Finding HVGs', end = "; ")
    # Restrict downstream steps to variable genes
    sc.pp.highly_variable_genes(adata)
    adata = adata[:, adata.var_names[adata.var['highly_variable']]].copy()

    print('Scaling', end = "; ")
    # Scale variable genes
    sc.pp.scale(adata)

    print('PCA', end = "; ")
    # Run PCA
    sc.tl.pca(adata, svd_solver = 'arpack')
    
    print('Neighbors', end = "; ")
    # Find nearest neighbors
    sc.pp.neighbors(
        adata, 
        n_neighbors = 50,
        n_pcs = 30
    )

    print('Leiden', end = "; ")
    # Find clusters
    sc.tl.leiden(
        adata, 
        resolution = resolution, 
        key_added = 'leiden_{r}'.format(r = resolution),
        n_iterations = 2
    )

    print('UMAP', end = "; ")
    # Run UMAP
    sc.tl.umap(adata, min_dist = 0.05)

    print('Ranking genes', end = "; ")
    sc.tl.rank_genes_groups(adata, 'leiden_{r}'.format(r = resolution), method='t-test')
    
    print('Renormalizing')
    adata = adata.raw.to_adata()
    adata.raw = adata

    # Normalize and log transform
    sc.pp.normalize_total(adata)
    sc.pp.log1p(adata)
    
    return adata

In [10]:
def process_harmonize_adata(adata, resolution = 2):
    
    # Keep a copy of the raw data
    #adata = adata.raw.to_adata()
    #adata.raw = adata
    adata.raw = adata

    print('Normalizing', end = "; ")
    # Normalize and log transform
    sc.pp.normalize_total(adata)
    sc.pp.log1p(adata)

    print('Finding HVGs', end = "; ")
    # Restrict downstream steps to variable genes
    sc.pp.highly_variable_genes(adata)
    adata = adata[:, adata.var_names[adata.var['highly_variable']]].copy()

    print('Scaling', end = "; ")
    # Scale variable genes
    sc.pp.scale(adata)

    print('PCA', end = "; ")
    # Run PCA
    sc.tl.pca(adata, svd_solver = 'arpack')


    # Perform the operations on the Anndata object
    print('Harmonize', end = "; ")
    sce.pp.harmony_integrate(adata, ["batch_id", "subject.subjectGuid"])

    print('Neighbors', end = "; ")
    # Find nearest neighbors
    sc.pp.neighbors(
        adata, 
        n_neighbors = 50,
        n_pcs = 30
    )

    print('Leiden', end = "; ")
    # Find clusters
    sc.tl.leiden(
        adata, 
        resolution = resolution, 
        key_added = 'leiden_harmony_{r}'.format(r = resolution),
        n_iterations = 2,
        #key_added="leiden_harmony"
    )

    print('UMAP', end = "; ")
    # Run UMAP
    sc.tl.umap(adata, min_dist = 0.05, init_pos="X_pca_harmony")

    print('Ranking genes', end = "; ")
    sc.tl.rank_genes_groups(adata, 'leiden_harmony_{r}'.format(r = resolution), method='t-test')
    
    print('Renormalizing')
    adata = adata.raw.to_adata()
    adata.raw = adata

    # Normalize and log transform
    sc.pp.normalize_total(adata)
    sc.pp.log1p(adata)
    
    return adata

In [11]:
def format_cell_type(cell_type):
    cell_type = re.sub('\\+', 'pos', cell_type)
    cell_type = re.sub('-', 'neg', cell_type)
    cell_type = re.sub(' ', '_', cell_type)
    return cell_type

In [12]:
def element_id(n = 3):
    import periodictable
    from random import randrange
    rand_el = []
    for i in range(n):
        el = randrange(0,118)
        rand_el.append(periodictable.elements[el].name)
    rand_str = '-'.join(rand_el)
    return rand_str

## Identify files for use in HISE

In [13]:
search_id = 'europium-niobium-americium'

Retrieve files stored in our HISE project store

In [14]:
ps_df = hisepy.list_files_in_project_store('UCSDCU_Y4')
ps_df = ps_df[['id', 'name']]

Filter for files from the previous notebook using our search_id

In [15]:
search_df = ps_df[ps_df['name'].str.contains(search_id)]
search_df = search_df.sort_values('name')

In [16]:
h5ad_df = search_df[search_df['name'].str.contains('.h5ad')]

In [17]:
h5ad_df

,id,name
378,f2ead10f-8e81-4853-859b-35983b4dcf2b,europium-niobium-americium/pbmc_set1_initial_q...
372,51e7bf39-bc45-498d-912c-010f5d14500f,europium-niobium-americium/pbmc_set2_initial_q...
375,11db53b2-6a6a-4afa-a8af-4dab697edf51,europium-niobium-americium/pbmc_set3_initial_q...
369,cbf71821-5e2d-4d32-8c63-16a945c32192,europium-niobium-americium/pbmc_set4_initial_q...


In [18]:
h5ad_uuids = {}
for i in range(h5ad_df.shape[0]):
    fn = h5ad_df['name'].tolist()[i]
    group_name = re.sub('.+pbmc_', '', fn)
    group_name = re.sub('_init.+', '', group_name)
    h5ad_uuids[group_name] = h5ad_df['id'].tolist()[i]

In [19]:
h5ad_uuids

{'set1': 'f2ead10f-8e81-4853-859b-35983b4dcf2b',
 'set2': '51e7bf39-bc45-498d-912c-010f5d14500f',
 'set3': '11db53b2-6a6a-4afa-a8af-4dab697edf51',
 'set4': 'cbf71821-5e2d-4d32-8c63-16a945c32192'}

## Download and sort .h5ad files

Sorting the .h5ad files by AIFI_L2 will make reading each cell type much faster by placing cells of the same type next to each other in the sparse matrix.

## Open connections to .h5ad files

Now that they're sorted, we can open these files with on-disk backing so we don't have to read the entire file at once.

In [20]:
h5ad_conn = {}
for group_name, uuid in h5ad_uuids.items():
    h5ad_conn[group_name] = read_adata_backed_uuid(uuid)

## Process each cell type

In [21]:
adata = read_adata_backed_uuid(list(h5ad_uuids.values())[0])
l3_types = adata.obs['AIFI_L3'].unique().tolist()
l3_types.sort()

In [22]:
h5ad_conn

{'set1': AnnData object with n_obs × n_vars = 1923811 × 33538 backed at '/home/jupyter/cache/f2ead10f-8e81-4853-859b-35983b4dcf2b/pbmc_set1_initial_qc_2024-05-10.h5ad'
     obs: 'barcodes', 'batch_id', 'cell_name', 'cell_uuid', 'chip_id', 'hto_barcode', 'hto_category', 'n_genes', 'n_mito_umis', 'n_reads', 'n_umis', 'original_barcodes', 'pbmc_sample_id', 'pool_id', 'well_id', 'sample.sampleKitGuid', 'cohort.cohortGuid', 'subject.subjectGuid', 'subject.biologicalSex', 'subject.race', 'subject.ethnicity', 'subject.birthYear', 'sample.visitName', 'sample.drawDate', 'sample.diseaseStatesRecordedAtVisit', 'sample.daysSinceFirstVisit', 'file.id', 'subset_grp', 'predicted_doublet', 'doublet_score', 'AIFI_L1', 'AIFI_L1_score', 'AIFI_L2', 'AIFI_L2_score', 'AIFI_L3', 'AIFI_L3_score', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_

In [23]:
l3_types

['ASDC',
 'Activated memory B cell',
 'Adaptive NK cell',
 'BaEoMaP cell',
 'C1Q+ CD16 monocyte',
 'CD14+ cDC2',
 'CD27+ effector B cell',
 'CD27- effector B cell',
 'CD4 MAIT',
 'CD56bright NK cell',
 'CD8 MAIT',
 'CD8aa',
 'CD95 memory B cell',
 'CLP cell',
 'CM CD4 T cell',
 'CM CD8 T cell',
 'CMP cell',
 'Core CD14 monocyte',
 'Core CD16 monocyte',
 'Core memory B cell',
 'Core naive B cell',
 'Core naive CD4 T cell',
 'Core naive CD8 T cell',
 'DN T cell',
 'Early memory B cell',
 'Erythrocyte',
 'GZMB+ Vd2 gdT',
 'GZMB- CD27+ EM CD4 T cell',
 'GZMB- CD27- EM CD4 T cell',
 'GZMK+ CD27+ EM CD8 T cell',
 'GZMK+ CD56dim NK cell',
 'GZMK+ Vd2 gdT',
 'GZMK+ memory CD4 Treg',
 'GZMK- CD27+ EM CD8 T cell',
 'GZMK- CD56dim NK cell',
 'HLA-DRhi cDC2',
 'IL1B+ CD14 monocyte',
 'ILC',
 'ISG+ CD14 monocyte',
 'ISG+ CD16 monocyte',
 'ISG+ CD56dim NK cell',
 'ISG+ MAIT',
 'ISG+ cDC2',
 'ISG+ memory CD4 T cell',
 'ISG+ memory CD8 T cell',
 'ISG+ naive B cell',
 'ISG+ naive CD4 T cell',
 'ISG+ na

In [28]:
l3_types_sub = l3_types[start:end]

l3_types_sub

['Core naive B cell',
 'Core naive CD4 T cell',
 'Core naive CD8 T cell',
 'DN T cell',
 'Early memory B cell',
 'Erythrocyte',
 'GZMB+ Vd2 gdT',
 'GZMB- CD27+ EM CD4 T cell',
 'GZMB- CD27- EM CD4 T cell',
 'GZMK+ CD27+ EM CD8 T cell',
 'GZMK+ CD56dim NK cell',
 'GZMK+ Vd2 gdT',
 'GZMK+ memory CD4 Treg',
 'GZMK- CD27+ EM CD8 T cell',
 'GZMK- CD56dim NK cell',
 'HLA-DRhi cDC2',
 'IL1B+ CD14 monocyte',
 'ILC',
 'ISG+ CD14 monocyte',
 'ISG+ CD16 monocyte',
 'ISG+ CD56dim NK cell',
 'ISG+ MAIT',
 'ISG+ cDC2',
 'ISG+ memory CD4 T cell',
 'ISG+ memory CD8 T cell',
 'ISG+ naive B cell',
 'ISG+ naive CD4 T cell',
 'ISG+ naive CD8 T cell',
 'Intermediate monocyte',
 'KLRB1+ memory CD4 Treg']

In [29]:
def read_l3_type(adata, cell_type):
    type_adata = adata[adata.obs['AIFI_L3'] == cell_type].to_memory()
    return type_adata

In [30]:
out_files = []

### Harmonize

In [31]:
%%time

for cell_type in l3_types_sub:
    print(cell_type)
    
    # Read data from each group for this type in parallel
    print('Loading data')
    type_adata_dict = {}

    with ThreadPoolExecutor(max_workers = 4) as executor:
        futures = {
            executor.submit(
                read_l3_type, 
                h5ad_conn[group_name], 
                cell_type): group_name 
            for group_name in h5ad_conn.keys()
        }
        for future in concurrent.futures.as_completed(futures):
            future_group = futures[future]
            type_adata_dict[future_group] = future.result()
    
    # If small, combine and process
    print('Combining and processing')
    type_adata = sc.concat(type_adata_dict)
    print(type_adata.shape)
    
    if(type_adata.obs['AIFI_L1'][0] == 'B cells'):
        print('Dropping Ig Genes for B cell clustering')
        type_adata = remove_ig_genes(type_adata)
        print(type_adata.shape)
    
    type_adata = process_harmonize_adata(type_adata)
    
    print('Saving processed data')
    out_type = format_cell_type(cell_type)
    out_file = 'output/preRA_cluster_harmonize_{c}_{d}.h5ad'.format(
        g = group_name,
        c = out_type,
        d = date.today()
    )
    type_adata.write_h5ad(out_file)
    out_files.append(out_file)

Core naive B cell
Loading data


/opt/conda/lib/python3.10/site-packages/anndata/_core/anndata.py:1179: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_sub[k] = df_sub[k].cat.remove_unused_categories()


Combining and processing
(391130, 33538)
Normalizing; Finding HVGs; Scaling; PCA; Harmonize; 

2024-05-25 22:11:05,692 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...
2024-05-25 22:12:29,327 - harmonypy - INFO - sklearn.KMeans initialization complete.
2024-05-25 22:12:31,015 - harmonypy - INFO - Iteration 1 of 10
2024-05-25 22:23:24,410 - harmonypy - INFO - Iteration 2 of 10
2024-05-25 22:34:38,451 - harmonypy - INFO - Iteration 3 of 10
2024-05-25 22:39:54,025 - harmonypy - INFO - Converged after 3 iterations


Neighbors; Leiden; UMAP; Ranking genes; WARNING: It seems you use rank_genes_groups on the raw count data. Please logarithmize your data before calling rank_genes_groups.


...

Harmonize; 

2024-05-26 10:38:12,065 - harmonypy - INFO - sklearn.KMeans initialization complete.
2024-05-26 10:38:12,131 - harmonypy - INFO - Iteration 1 of 10
2024-05-26 10:38:16,773 - harmonypy - INFO - Iteration 2 of 10
2024-05-26 10:38:21,036 - harmonypy - INFO - Iteration 3 of 10
2024-05-26 10:38:24,821 - harmonypy - INFO - Iteration 4 of 10
2024-05-26 10:38:27,856 - harmonypy - INFO - Iteration 5 of 10
2024-05-26 10:38:30,499 - harmonypy - INFO - Iteration 6 of 10
2024-05-26 10:38:33,384 - harmonypy - INFO - Iteration 7 of 10
2024-05-26 10:38:36,303 - harmonypy - INFO - Iteration 8 of 10
2024-05-26 10:38:38,784 - harmonypy - INFO - Iteration 9 of 10
2024-05-26 10:38:41,447 - harmonypy - INFO - Iteration 10 of 10
2024-05-26 10:38:43,907 - harmonypy - INFO - Stopped before convergence


Neighbors; Leiden; UMAP; Ranking genes; WARNING: It seems you use rank_genes_groups on the raw count data. Please logarithmize your data before calling rank_genes_groups.


/opt/conda/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:396: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, 'names'] = self.var_names[global_indices]
/opt/conda/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:398: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, 'scores'] = scores[global_indices]
/opt/conda/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:401: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result

Renormalizing
Saving processed data
CPU times: user 3d 20h 16min 55s, sys: 3d 22h 24min 53s, total: 7d 18h 41min 48s
Wall time: 12h 29min 30s


In [54]:
type_adata

AnnData object with n_obs × n_vars = 1619213 × 33538
    obs: 'barcodes', 'batch_id', 'cell_name', 'cell_uuid', 'chip_id', 'hto_barcode', 'hto_category', 'n_genes', 'n_mito_umis', 'n_reads', 'n_umis', 'original_barcodes', 'pbmc_sample_id', 'pool_id', 'well_id', 'sample.sampleKitGuid', 'cohort.cohortGuid', 'subject.subjectGuid', 'subject.biologicalSex', 'subject.race', 'subject.ethnicity', 'subject.birthYear', 'sample.visitName', 'sample.drawDate', 'sample.diseaseStatesRecordedAtVisit', 'sample.daysSinceFirstVisit', 'file.id', 'subset_grp', 'predicted_doublet', 'doublet_score', 'AIFI_L1', 'AIFI_L1_score', 'AIFI_L2', 'AIFI_L2_score', 'AIFI_L3', 'AIFI_L3_score', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mito', 'log1p_total_counts_mito', 'pct_counts_mito'
    var: 'highly_variable', 'means', 'dispersions', 'disp

In [33]:
out_files

['output/preRA_cluster_harmonize_Core_naive_B_cell_2024-05-25.h5ad',
 'output/preRA_cluster_harmonize_Core_naive_CD4_T_cell_2024-05-26.h5ad',
 'output/preRA_cluster_harmonize_Core_naive_CD8_T_cell_2024-05-26.h5ad',
 'output/preRA_cluster_harmonize_DN_T_cell_2024-05-26.h5ad',
 'output/preRA_cluster_harmonize_Early_memory_B_cell_2024-05-26.h5ad',
 'output/preRA_cluster_harmonize_Erythrocyte_2024-05-26.h5ad',
 'output/preRA_cluster_harmonize_GZMBpos_Vd2_gdT_2024-05-26.h5ad',
 'output/preRA_cluster_harmonize_GZMBneg_CD27pos_EM_CD4_T_cell_2024-05-26.h5ad',
 'output/preRA_cluster_harmonize_GZMBneg_CD27neg_EM_CD4_T_cell_2024-05-26.h5ad',
 'output/preRA_cluster_harmonize_GZMKpos_CD27pos_EM_CD8_T_cell_2024-05-26.h5ad',
 'output/preRA_cluster_harmonize_GZMKpos_CD56dim_NK_cell_2024-05-26.h5ad',
 'output/preRA_cluster_harmonize_GZMKpos_Vd2_gdT_2024-05-26.h5ad',
 'output/preRA_cluster_harmonize_GZMKpos_memory_CD4_Treg_2024-05-26.h5ad',
 'output/preRA_cluster_harmonize_GZMKneg_CD27pos_EM_CD8_T_cell_

In [34]:
adata

AnnData object with n_obs × n_vars = 1923811 × 33538 backed at '/home/jupyter/cache/f2ead10f-8e81-4853-859b-35983b4dcf2b/pbmc_set1_initial_qc_2024-05-10.h5ad'
    obs: 'barcodes', 'batch_id', 'cell_name', 'cell_uuid', 'chip_id', 'hto_barcode', 'hto_category', 'n_genes', 'n_mito_umis', 'n_reads', 'n_umis', 'original_barcodes', 'pbmc_sample_id', 'pool_id', 'well_id', 'sample.sampleKitGuid', 'cohort.cohortGuid', 'subject.subjectGuid', 'subject.biologicalSex', 'subject.race', 'subject.ethnicity', 'subject.birthYear', 'sample.visitName', 'sample.drawDate', 'sample.diseaseStatesRecordedAtVisit', 'sample.daysSinceFirstVisit', 'file.id', 'subset_grp', 'predicted_doublet', 'doublet_score', 'AIFI_L1', 'AIFI_L1_score', 'AIFI_L2', 'AIFI_L2_score', 'AIFI_L3', 'AIFI_L3_score', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mit

## Upload Cell Type data to HISE

Finally, we'll use `hisepy.upload.upload_files()` to send a copy of our output to HISE to use for downstream analysis steps.

In [45]:
study_space_uuid = '223de760-9624-45bd-aefe-ca24c75b1800'
title = 'IDE for 05 PBMC L3 Pre-cleanup Harmonized Clustering 2/3 {d}'.format(d = date.today())
print(title)

IDE for 05 PBMC L3 Pre-cleanup Harmonized Clustering 2/3 2024-06-03


In [46]:
### preserve search id from script 5a
search_id = 'polonium-strontium-zirconium'
search_id

'polonium-strontium-zirconium'

In [38]:
in_files = list(h5ad_uuids.values())
in_files

['f2ead10f-8e81-4853-859b-35983b4dcf2b',
 '51e7bf39-bc45-498d-912c-010f5d14500f',
 '11db53b2-6a6a-4afa-a8af-4dab697edf51',
 'cbf71821-5e2d-4d32-8c63-16a945c32192']

In [39]:
out_files

['output/preRA_cluster_harmonize_Core_naive_B_cell_2024-05-25.h5ad',
 'output/preRA_cluster_harmonize_Core_naive_CD4_T_cell_2024-05-26.h5ad',
 'output/preRA_cluster_harmonize_Core_naive_CD8_T_cell_2024-05-26.h5ad',
 'output/preRA_cluster_harmonize_DN_T_cell_2024-05-26.h5ad',
 'output/preRA_cluster_harmonize_Early_memory_B_cell_2024-05-26.h5ad',
 'output/preRA_cluster_harmonize_Erythrocyte_2024-05-26.h5ad',
 'output/preRA_cluster_harmonize_GZMBpos_Vd2_gdT_2024-05-26.h5ad',
 'output/preRA_cluster_harmonize_GZMBneg_CD27pos_EM_CD4_T_cell_2024-05-26.h5ad',
 'output/preRA_cluster_harmonize_GZMBneg_CD27neg_EM_CD4_T_cell_2024-05-26.h5ad',
 'output/preRA_cluster_harmonize_GZMKpos_CD27pos_EM_CD8_T_cell_2024-05-26.h5ad',
 'output/preRA_cluster_harmonize_GZMKpos_CD56dim_NK_cell_2024-05-26.h5ad',
 'output/preRA_cluster_harmonize_GZMKpos_Vd2_gdT_2024-05-26.h5ad',
 'output/preRA_cluster_harmonize_GZMKpos_memory_CD4_Treg_2024-05-26.h5ad',
 'output/preRA_cluster_harmonize_GZMKneg_CD27pos_EM_CD8_T_cell_

In [47]:
hisepy.upload.upload_files(
    files = out_files,
    study_space_id = study_space_uuid,
    title = title,
    input_file_ids = in_files,
    destination = search_id
)

you are trying to upload file_ids... ['output/preRA_cluster_harmonize_Core_naive_B_cell_2024-05-25.h5ad', 'output/preRA_cluster_harmonize_Core_naive_CD4_T_cell_2024-05-26.h5ad', 'output/preRA_cluster_harmonize_Core_naive_CD8_T_cell_2024-05-26.h5ad', 'output/preRA_cluster_harmonize_DN_T_cell_2024-05-26.h5ad', 'output/preRA_cluster_harmonize_Early_memory_B_cell_2024-05-26.h5ad', 'output/preRA_cluster_harmonize_Erythrocyte_2024-05-26.h5ad', 'output/preRA_cluster_harmonize_GZMBpos_Vd2_gdT_2024-05-26.h5ad', 'output/preRA_cluster_harmonize_GZMBneg_CD27pos_EM_CD4_T_cell_2024-05-26.h5ad', 'output/preRA_cluster_harmonize_GZMBneg_CD27neg_EM_CD4_T_cell_2024-05-26.h5ad', 'output/preRA_cluster_harmonize_GZMKpos_CD27pos_EM_CD8_T_cell_2024-05-26.h5ad', 'output/preRA_cluster_harmonize_GZMKpos_CD56dim_NK_cell_2024-05-26.h5ad', 'output/preRA_cluster_harmonize_GZMKpos_Vd2_gdT_2024-05-26.h5ad', 'output/preRA_cluster_harmonize_GZMKpos_memory_CD4_Treg_2024-05-26.h5ad', 'output/preRA_cluster_harmonize_GZMKne

(y/n) y


{'trace_id': '1a129391-d343-46fb-bb52-57dd8f213566',
 'files': ['output/preRA_cluster_harmonize_Core_naive_B_cell_2024-05-25.h5ad',
  'output/preRA_cluster_harmonize_Core_naive_CD4_T_cell_2024-05-26.h5ad',
  'output/preRA_cluster_harmonize_Core_naive_CD8_T_cell_2024-05-26.h5ad',
  'output/preRA_cluster_harmonize_DN_T_cell_2024-05-26.h5ad',
  'output/preRA_cluster_harmonize_Early_memory_B_cell_2024-05-26.h5ad',
  'output/preRA_cluster_harmonize_Erythrocyte_2024-05-26.h5ad',
  'output/preRA_cluster_harmonize_GZMBpos_Vd2_gdT_2024-05-26.h5ad',
  'output/preRA_cluster_harmonize_GZMBneg_CD27pos_EM_CD4_T_cell_2024-05-26.h5ad',
  'output/preRA_cluster_harmonize_GZMBneg_CD27neg_EM_CD4_T_cell_2024-05-26.h5ad',
  'output/preRA_cluster_harmonize_GZMKpos_CD27pos_EM_CD8_T_cell_2024-05-26.h5ad',
  'output/preRA_cluster_harmonize_GZMKpos_CD56dim_NK_cell_2024-05-26.h5ad',
  'output/preRA_cluster_harmonize_GZMKpos_Vd2_gdT_2024-05-26.h5ad',
  'output/preRA_cluster_harmonize_GZMKpos_memory_CD4_Treg_2024-0

In [48]:
import session_info
session_info.show()